# 02 — Schema Detection
**Purpose:** Confirm every file uses the same format before writing any analysis code.

A CSV file can use different delimiters (comma, semicolon, tab), encodings
(UTF-8, Latin-1), and column structures. Getting this wrong means reading garbage.

**This notebook answers:**
- What delimiter is used? (semicolon, not comma)
- What encoding? (UTF-8)
- Do all 233 files follow the same schema?
- How many rows does each file have (estimated)?

In [ ]:
import sys, csv, io
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from zoro_eda.config import load_config
from zoro_eda.paths import resolve_paths
from zoro_eda.csv_io import detect_encoding, detect_delimiter, find_column, estimate_row_count

cfg = load_config()
paths = resolve_paths(cfg=cfg)
print("Libraries loaded.")

## Step 1: Look at raw bytes first

Before writing any parsing code, we look at the raw bytes of one file.
This is always the right starting point — never assume the format.

In [ ]:
sample_file = paths.raw_data / "real_BatterieLeistung.csv"

with open(sample_file, "r", encoding="utf-8") as fh:
    for i, line in enumerate(fh):
        print(repr(line.rstrip()))
        if i >= 4:
            break

### What we see

```
'Unnamed: 0;_time;_value;_field;_measurement'
';2022-12-07T14:11:41Z;6.43;value;real_BatterieLeistung'
```

Key observations:
- **Separator is `;` (semicolon)** — Excel would misread this as one column
- **Column order:** `Unnamed:0 | _time | _value | _field | _measurement`
- **`Unnamed: 0`** is always empty — InfluxDB export artifact (ignore it)
- **`_time`** is ISO 8601 UTC (`Z` = UTC timezone)
- **`_value`** is a float
- **`_field`** is always `value` — single-field measurement
- **`_measurement`** is the signal name (matches the filename stem)

This is the standard **InfluxDB CSV export format**.

## Step 2: Delimiter auto-detection

`detect_delimiter()` from our shared library tries `; , \t |` on the
first line and picks whichever gives the most columns.

This is robust even if a future file uses a different delimiter.

In [ ]:
with open(sample_file, "rb") as fh:
    raw_bytes = fh.read(6_144)  # 6 KB — enough for header + ~20 rows

detected_encoding = detect_encoding(raw_bytes)
detected_delimiter, detected_columns = detect_delimiter(raw_bytes)

print(f"Encoding   : {detected_encoding}")
print(f"Delimiter  : {detected_delimiter!r}")
print(f"Columns    : {detected_columns}")
print(f"Num columns: {len(detected_columns)}")

## Step 3: Check schema consistency across all 233 files

We read only 6 KB per file. This is fast enough for all 233 files without
approaching the 40 GB data limit.

In [ ]:
csv_files = sorted([f for f in paths.raw_data.iterdir()
                   if f.is_file() and f.suffix.lower() == ".csv"])

results = []
for fpath in csv_files:
    with open(fpath, "rb") as fh:
        raw = fh.read(6_144)
    enc = detect_encoding(raw)
    delim, cols = detect_delimiter(raw)
    has_time  = find_column(cols, "_time") >= 0
    has_value = find_column(cols, "_value") >= 0
    has_meas  = find_column(cols, "_measurement") >= 0
    schema = "standard" if (has_time and has_value and has_meas) else "non-standard"
    est_rows = estimate_row_count(fpath)
    results.append({"file": fpath.name, "encoding": enc,
                    "delimiter": delim, "schema": schema, "est_rows": est_rows})

from collections import Counter
print(f"Schema types : {dict(Counter(r['schema'] for r in results))}")
print(f"Encodings    : {dict(Counter(r['encoding'] for r in results))}")
print(f"Delimiters   : {dict(Counter(r['delimiter'] for r in results))}")

## Step 4: Estimated row counts

We estimate rows from file size ÷ average row byte size (measured from first 64 KB).
This avoids reading the full 40 GB but gives a good approximation.

In [ ]:
top_by_rows = sorted(results, key=lambda r: r["est_rows"] or 0, reverse=True)
print(f"{'File':<50} {'Est. rows':>13}")
print("-" * 65)
for r in top_by_rows[:10]:
    print(f"{r['file']:<50} {r['est_rows']:>13,}")

## Key findings

| Finding | Value |
|---|---|
| All 233 files | Standard InfluxDB schema |
| Encoding | UTF-8 (100%) |
| Delimiter | Semicolon `;` (100%) |
| Max estimated rows | ~7 million (green_t1/t2/t3) |
| Min rows | 6 rows (commissioning snapshots) |

**100% format consistency** is unusual in real-world BMS data. It means
we can write one parser for all signals — no per-file special cases.

**Next:** `03_timeseries_profiling.ipynb` — when does each signal start and end?